# Estudio 06 · Fase 2 · Calibración sobre L

Preregistro en `code/preregistration.py → PHASE2` (commit `3bbfa10`, aprobado por Rosa sin cambios antes de
calibrar ninguna variante).

- **Base:** las predicciones sin calibrar de L guardadas en la 1a (`fase1a/predicciones_L.csv.gz`, 5 semillas).
  No se reentrena nada; μ no cambia.
- **Referencia:** Platt por horizonte con recorte del logit 1e-6.
- **Paso A:** recorte 1e-15 frente a 1e-6. **Paso B** (con el recorte del paso A): Platt agrupado, beta e
  isotónica (≥ 500 positivos; por debajo, Platt) frente a Platt por horizonte.
- **Regla:** 5 temporadas (2010, 2015, 2016, 2022, 2023), AUPRC veta con 2 de 5, guarda de calibración; se
  adopta con «gana» en ≥ 2 de 3 horizontes, sin «pierde» ni «compensa». 2010 aparte; 2024 contraste.


In [1]:
import json
import sys

import pandas as pd

from denguekay.config.paths import RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.model import tracking

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase2_calibration as p2
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "fase2.json"
preds = p2.read_l_predictions()
series = p2.labels_series()
{s: len(p) for s, p in preds.items()}


{17: 183105, 29: 183105, 41: 183105, 53: 183105, 67: 183105}

## 1 · Guardas antes de mirar resultados (G5; G6 tras calibrar; G7 es un test)


In [2]:
g5 = {s: p2.reproduces_protocol(p) for s, p in preds.items()}
if not all(v["pasa"] for v in g5.values()):
    raise RuntimeError(f"Falla G5: se detiene la fase. {g5}")
g5


{17: {'pasa': True, 'diferencia_maxima': 0.0},
 29: {'pasa': True, 'diferencia_maxima': 0.0},
 41: {'pasa': True, 'diferencia_maxima': 0.0},
 53: {'pasa': True, 'diferencia_maxima': 0.0},
 67: {'pasa': True, 'diferencia_maxima': 0.0}}

## 2 · Paso A · recorte del logit


In [3]:
ref = p2.run_variant(preds, series, "platt", pr.PHASE2_EPS_REFERENCE)
small = p2.run_variant(preds, series, "platt", pr.PHASE2_EPS_SMALL)
g6 = {"platt_1e-6": sum(p2.monotone_violations(e["datos"]) for e in ref["evaluaciones"].values()),
      "platt_1e-15": sum(p2.monotone_violations(e["datos"]) for e in small["evaluaciones"].values())}
if any(g6.values()):
    raise RuntimeError(f"Falla G6: se detiene la fase. {g6}")
step_a = p2.compare(small, ref)
guard_a = {"referencia": p2.calibration_against(ref), "recorte_1e-15": p2.calibration_against(small)}
adopt_a = pr.adopt_phase2(step_a["veredicto"], guard_a["recorte_1e-15"])
eps = pr.PHASE2_EPS_SMALL if adopt_a else pr.PHASE2_EPS_REFERENCE
print("Adopta 1e-15:", adopt_a, "| recorte del paso B:", eps)
step_a["veredicto"]


Adopta 1e-15: False | recorte del paso B: 1e-06


,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,False,False,NaN,None,0,0,0,0,0,0,2,0,0,0
1,3,empate,False,False,NaN,None,0,0,0,0,2,0,2,0,0,0
2,4,empate,False,False,brier,None,0,0,0,0,2,0,3,0,0,0


In [4]:
display(step_a["antelacion"].round(3))
display(pd.DataFrame(guard_a))
step_a["metricas"].round(4)


,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2010,diferencia_temporada_2015,diferencia_temporada_2016,diferencia_temporada_2022,diferencia_temporada_2023,episodios_temporada_2010,episodios_temporada_2015,episodios_temporada_2016,episodios_temporada_2022,episodios_temporada_2023
0,2,193,0.000,0.000,0.0,0.000,0.0,0.000,0.000,0.000,30,34,34,38,57
1,3,193,-0.005,-0.016,0.0,0.000,0.0,-0.029,0.000,0.000,30,34,34,38,57
2,4,193,-0.016,-0.036,0.0,-0.033,0.0,0.000,-0.026,-0.018,30,34,34,38,57


,referencia,recorte_1e-15
horizonte,,
2,1,1
3,1,1
4,1,1


,horizonte,bloque,positivos_cand,f1_050_cand,f1_050_ref,recall_050_cand,recall_050_ref,precision_050_cand,precision_050_ref,auprc_cand,auprc_ref,brier_cand,brier_ref,log_loss_cand,log_loss_ref,brier_sin_calibrar_cand,brier_sin_calibrar_ref,log_loss_sin_calibrar_cand,log_loss_sin_calibrar_ref
3,2,temporada_2010,234.0,0.7113,0.7225,0.6667,0.7009,0.7633,0.7455,0.7804,0.7804,0.0304,0.0303,0.1102,0.1095,0.0325,0.0325,0.1142,0.1142
8,2,temporada_2015,557.0,0.7658,0.7725,0.6858,0.6948,0.8685,0.8652,0.8845,0.8845,0.0514,0.0510,0.1742,0.1735,0.0646,0.0646,0.2035,0.2035
9,2,temporada_2016,395.0,0.6581,0.6573,0.7139,0.7139,0.6104,0.6091,0.7566,0.7566,0.0592,0.0594,0.1930,0.1933,0.0554,0.0554,0.1848,0.1848
15,2,temporada_2022,431.0,0.7119,0.7119,0.6961,0.6961,0.7311,0.7282,0.7808,0.7808,0.0537,0.0537,0.1812,0.1813,0.0547,0.0547,0.1832,0.1832
16,2,temporada_2023,1039.0,0.8579,0.8574,0.8268,0.8277,0.8904,0.8885,0.9475,0.9475,0.0619,0.0618,0.2070,0.2067,0.0712,0.0712,0.2353,0.2353
17,2,temporada_2024,857.0,0.7998,0.7993,0.8530,0.8530,0.7528,0.7521,0.8935,0.8935,0.0736,0.0736,0.2317,0.2318,0.0743,0.0743,0.2384,0.2384
21,3,temporada_2010,234.0,0.6067,0.6308,0.5043,0.5513,0.7697,0.7318,0.7058,0.7058,0.0364,0.0360,0.1302,0.1288,0.0390,0.0390,0.1342,0.1342
26,3,temporada_2015,557.0,0.6344,0.6486,0.4937,0.5153,0.8850,0.8750,0.8391,0.8391,0.0651,0.0642,0.2122,0.2109,0.0809,0.0809,0.2467,0.2467
27,3,temporada_2016,395.0,0.6271,0.6273,0.6532,0.6582,0.6009,0.5991,0.7093,0.7093,0.0652,0.0654,0.2147,0.2150,0.0635,0.0635,0.2095,0.2095
33,3,temporada_2022,431.0,0.7012,0.7043,0.6752,0.6821,0.7312,0.7277,0.7710,0.7710,0.0561,0.0561,0.1901,0.1901,0.0596,0.0596,0.1978,0.1978


## 3 · Paso B · familias de calibrador (con el recorte del paso A)


In [5]:
ref_b = small if adopt_a else ref
variants = {"platt": ref_b}
for name in ("platt_agrupado", "beta", "isotonica"):
    method, pooled = p2.VARIANTS[name]
    variants[name] = p2.run_variant(preds, series, method, eps, pooled)
g6.update({name: sum(p2.monotone_violations(e["datos"]) for e in variants[name]["evaluaciones"].values())
           for name in ("platt_agrupado", "beta")})
if any(g6.values()):
    raise RuntimeError(f"Falla G6: se detiene la fase. {g6}")
step_b, guard_b, adopted = {}, {"platt": p2.calibration_against(ref_b)}, {}
for name in ("platt_agrupado", "beta", "isotonica"):
    step_b[name] = p2.compare(variants[name], ref_b)
    guard_b[name] = p2.calibration_against(variants[name])
    if pr.adopt_phase2(step_b[name]["veredicto"], guard_b[name]):
        adopted[name] = variants[name]
final, crosses = p2.final_choice(adopted, "platt")
print("G6:", g6)
print("Adoptadas en el paso B:", list(adopted), "| cruces:", crosses)
print("Calibrador final:", final, "con recorte", eps)
pd.concat({k: v["veredicto"] for k, v in step_b.items()}, names=["candidata"]).reset_index(level=0)


G6: {'platt_1e-6': 0, 'platt_1e-15': 0, 'platt_agrupado': 0, 'beta': 0}
Adoptadas en el paso B: [] | cruces: []
Calibrador final: platt con recorte 1e-06


,candidata,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,platt_agrupado,2,empate,False,False,None,None,1,0,1,0,0,0,2,0,2,0
1,platt_agrupado,3,empate,False,False,None,None,0,0,0,0,0,0,1,0,0,0
2,platt_agrupado,4,empate,False,False,None,None,0,0,1,0,2,0,1,0,0,1
0,beta,2,empate,False,False,None,brier,1,0,5,0,0,0,0,0,0,1
1,beta,3,empate,False,False,None,brier,1,0,4,0,0,0,0,0,0,1
2,beta,4,empate,False,False,None,brier,3,0,4,0,0,0,0,0,2,0
0,isotonica,2,empate,False,False,auprc,brier,0,0,3,0,2,4,1,0,0,1
1,isotonica,3,empate,False,False,auprc,brier,0,0,3,0,0,4,0,0,0,1
2,isotonica,4,empate,False,False,auprc,NaN,2,0,2,0,0,4,0,0,3,0


In [6]:
display(pd.DataFrame(guard_b))
display(pd.concat({k: v["antelacion"] for k, v in step_b.items()}, names=["candidata"]).round(3))
pd.concat({k: v["metricas"] for k, v in step_b.items()}, names=["candidata"]).round(4)


,platt,platt_agrupado,beta,isotonica
horizonte,,,,
2,1,1,1,2
3,1,1,1,1
4,1,0,0,2


horizonte  episodios  diferencia  ic95_inferior  ic95_superior  diferencia_temporada_2010  diferencia_temporada_2015  diferencia_temporada_2016  diferencia_temporada_2022  diferencia_temporada_2023  \
candidata                                                                                                                                                                                                                 
platt_agrupado 0          2        193       0.047          0.021          0.078                        0.0                      0.000                      0.029                      0.079                      0.088   
               1          3        193       0.000          0.000          0.000                        0.0                      0.000                      0.000                      0.000                      0.000   
               2          4        193      -0.021         -0.041         -0.005                        0.0                      0.000                      0.000                     -0.026                     -0.053   
beta           0          2        193      -0.021         -0.041         -0.005                        0.0                      0.000                     -0.029                     -0.026                     -0.035   
               1          3        193      -0.005         -0.026          0.010                        0.0                      0.029                     -0.059                      0.000                      0.000   
               2          4        193       0.036          0.010          0.062                        0.0                      0.029                      0.029                      0.053                      0.053   
isotonica      0          2        193      -0.031         -0.057         -0.010                        0.0                     -0.029                     -0.029                     -0.026                     -0.053   
               1          3        193      -0.010         -0.026          0.000                        0.0                      0.000                     -0.059                      0.000                      0.000   
               2          4        193       0.036          0.010          0.062                        0.0                      0.000                      0.059                      0.053                      0.053   

                  episodios_temporada_2010  episodios_temporada_2015  episodios_temporada_2016  episodios_temporada_2022  episodios_temporada_2023  
candidata                                                                                                                                           
platt_agrupado 0                        30                        34                        34                        38                        57  
               1                        30                        34                        34                        38                        57  
               2                        30                        34                        34                        38                        57  
beta           0                        30                        34                        34                        38                        57  
               1                        30                        34                        34                        38                        57  
               2                        30                        34                        34                        38                        57  
isotonica      0                        30                        34                        34                        38                        57  
               1                        30                        34                        34                        38                        57  
               2                        30                        34                        34                        

horizonte          bloque  positivos_cand  f1_050_cand  f1_050_ref  recall_050_cand  recall_050_ref  precision_050_cand  precision_050_ref  auprc_cand  auprc_ref  brier_cand  brier_ref  \
candidata                                                                                                                                                                                                     
platt_agrupado 3           2  temporada_2010           234.0       0.7196      0.7225           0.6966          0.7009              0.7444             0.7455      0.7804     0.7804      0.0304     0.0303   
               8           2  temporada_2015           557.0       0.7709      0.7725           0.6930          0.6948              0.8671             0.8652      0.8845     0.8845      0.0510     0.0510   
               9           2  temporada_2016           395.0       0.6560      0.6573           0.7241          0.7139              0.5996             0.6091      0.7566     0.7566      0.0608     0.0594   
               15          2  temporada_2022           431.0       0.7260      0.7119           0.7378          0.6961              0.7146             0.7282      0.7808     0.7808      0.0543     0.0537   
               16          2  temporada_2023          1039.0       0.8609      0.8574           0.8527          0.8277              0.8690             0.8885      0.9475     0.9475      0.0612     0.0618   
               17          2  temporada_2024           857.0       0.7955      0.7993           0.8681          0.8530              0.7359             0.7521      0.8935     0.8935      0.0756     0.0736   
               21          3  temporada_2010           234.0       0.6344      0.6308           0.5598          0.5513              0.7318             0.7318      0.7058     0.7058      0.0360     0.0360   
               26          3  temporada_2015           557.0       0.6463      0.6486           0.5117          0.5153              0.8773             0.8750      0.8391     0.8391      0.0644     0.0642   
               27          3  temporada_2016           395.0       0.6279      0.6273           0.6557          0.6582              0.6005             0.5991      0.7093     0.7093      0.0652     0.0654   
               33          3  temporada_2022           431.0       0.7012      0.7043           0.6752          0.6821              0.7312             0.7277      0.7710     0.7710      0.0561     0.0561   
               34          3  temporada_2023          1039.0       0.8325      0.8350           0.7892          0.7940              0.8808             0.8803      0.9303     0.9303      0.0732     0.0730   
               35          3  temporada_2024           857.0       0.7878      0.7878           0.8623          0.8635              0.7271             0.7252      0.8762     0.8762      0.0793     0.0795   
               39          4  temporada_2010           234.0       0.4986      0.4759           0.3846          0.3590              0.7131             0.7143      0.5997     0.5997      0.0418     0.0419   
               44          4  temporada_2015           557.0       0.4796      0.4796           0.3268          0.3268              0.9010             0.9010      0.8109     0.8109      0.0747     0.0747   
               45          4  temporada_2016           395.0       0.5914      0.5904           0.5570          0.5722              0.6278             0.6081      0.6596     0.6596      0.0687     0.0695   
               51          4  temporada_2022           431.0       0.6386      0.6524           0.5638          0.5986              0.7365             0.7179      0.7425     0.7425      0.0597     0.0596   
               52          4  temporada_2023          1039.0       0.8011      0.8108           0.7170          0.7363              0.9091             0.9012      0.9220     0.9220      0.0840     0.0809   
               53          4  temporada_2024           857.0       0.7664      0.7610 

In [7]:
# Parámetros y respaldos de los calibradores (semilla 17)
cal = pd.concat({k: v["calibradores"][v["calibradores"].semilla.eq(pr.SEEDS[0])] for k, v in variants.items()},
                names=["variante"]).reset_index(level=0)
cal = cal[cal.bloque.isin([pr.block_name(s) for s in (*pr.DECIDING_SEASONS, pr.CONTRAST_SEASON)])]
cal.drop(columns=["corte", "semilla"]).round(4)


,variante,horizonte,bloque,metodo,eps,agrupado,filas_calibracion,positivos_calibracion,a,c,b,terminos,respaldo,escalones
2,platt,2,temporada_2010,platt,0.0,False,6760,258,1.0240,0.7282,NaN,NaN,NaN,NaN
7,platt,2,temporada_2015,platt,0.0,False,23660,709,1.1826,0.7446,NaN,NaN,NaN,NaN
8,platt,2,temporada_2016,platt,0.0,False,27105,1258,1.2332,1.0116,NaN,NaN,NaN,NaN
14,platt,2,temporada_2022,platt,0.0,False,47450,2353,1.2086,0.6366,NaN,NaN,NaN,NaN
15,platt,2,temporada_2023,platt,0.0,False,50830,2788,1.1967,0.5986,NaN,NaN,NaN,NaN
16,platt,2,temporada_2024,platt,0.0,False,54210,3793,1.2253,0.6836,NaN,NaN,NaN,NaN
20,platt,3,temporada_2010,platt,0.0,False,6695,257,1.0048,0.7176,NaN,NaN,NaN,NaN
25,platt,3,temporada_2015,platt,0.0,False,23595,704,1.1946,0.7611,NaN,NaN,NaN,NaN
26,platt,3,temporada_2016,platt,0.0,False,27040,1247,1.2589,1.1316,NaN,NaN,NaN,NaN
32,platt,3,temporada_2022,platt,0.0,False,47385,2346,1.2494,0.7794,NaN,NaN,NaN,NaN


## 4 · Lecturas que no deciden


In [8]:
no_2010 = (2015, 2016, 2022, 2023)
reading_no_2010 = {k: p2.compare(variants[k], ref_b, seasons=no_2010, min_in_favour=3,
                                 guard_veto=pr.GUARD_VETO_AGAINST)["veredicto"]
                   for k in ("platt_agrupado", "beta", "isotonica")}
reading_no_2010["recorte_1e-15"] = p2.compare(small, ref, seasons=no_2010, min_in_favour=3,
                                             guard_veto=pr.GUARD_VETO_AGAINST)["veredicto"]
pd.concat(reading_no_2010, names=["candidata"]).reset_index(level=0)


,candidata,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,platt_agrupado,2,empate,False,False,None,None,1,0,1,0,0,0,2,0,2,0
1,platt_agrupado,3,empate,False,False,None,None,0,0,0,0,0,0,1,0,0,0
2,platt_agrupado,4,empate,False,False,None,None,0,0,1,0,2,0,1,0,0,1
0,beta,2,empate,False,False,None,brier,1,0,4,0,0,0,0,0,0,1
1,beta,3,empate,False,False,None,brier,1,0,3,0,0,0,0,0,0,1
2,beta,4,empate,False,False,None,brier,2,0,3,0,0,0,0,0,2,0
0,isotonica,2,empate,False,False,auprc,brier,0,0,3,0,2,4,1,0,0,1
1,isotonica,3,empate,False,False,auprc,brier,0,0,3,0,0,4,0,0,0,1
2,isotonica,4,compensa (empate),True,False,auprc,NaN,2,0,2,0,0,4,0,0,3,0
0,recorte_1e-15,2,empate,False,False,None,None,0,0,0,0,0,0,1,0,0,0


In [9]:
rd = {k: p2.readings(v, series) for k, v in {"referencia": ref, **{n: variants[n] for n in variants if n != "platt"},
                                                  "recorte_1e-15": small}.items()}
cols = ["horizonte", "bloque", "positivos", "f1_050", "recall_050", "precision_050", "auprc", "brier",
        "brier_sin_calibrar", "log_loss", "log_loss_sin_calibrar"]
pd.concat({k: v["metricas_por_temporada"][cols] for k, v in rd.items()}, names=["variante"]).round(3)


horizonte          bloque  positivos  f1_050  recall_050  precision_050  auprc  brier  brier_sin_calibrar  log_loss  log_loss_sin_calibrar
variante                                                                                                                                                     
referencia     3           2  temporada_2010      234.0   0.722       0.701          0.745  0.780  0.030               0.033     0.110                  0.114
               8           2  temporada_2015      557.0   0.772       0.695          0.865  0.885  0.051               0.065     0.174                  0.203
               9           2  temporada_2016      395.0   0.657       0.714          0.609  0.757  0.059               0.055     0.193                  0.185
               10          2  temporada_2017      539.0   0.703       0.744          0.664  0.800  0.068               0.069     0.227                  0.229
               15          2  temporada_2022      431.0   0.712       0.696          0.728  0.781  0.054               0.055     0.181                  0.183
               16          2  temporada_2023     1039.0   0.857       0.828          0.889  0.947  0.062               0.071     0.207                  0.235
               17          2  temporada_2024      857.0   0.799       0.853          0.752  0.893  0.074               0.074     0.232                  0.238
               21          3  temporada_2010      234.0   0.631       0.551          0.732  0.706  0.036               0.039     0.129                  0.134
               26          3  temporada_2015      557.0   0.649       0.515          0.875  0.839  0.064               0.081     0.211                  0.247
               27          3  temporada_2016      395.0   0.627       0.658          0.599  0.709  0.065               0.063     0.215                  0.210
               28          3  temporada_2017      539.0   0.658       0.664          0.652  0.738  0.076               0.081     0.250                  0.258
               33          3  temporada_2022      431.0   0.704       0.682          0.728  0.771  0.056               0.060     0.190                  0.198
               34          3  temporada_2023     1039.0   0.835       0.794          0.880  0.930  0.073               0.093     0.245                  0.295
               35          3  temporada_2024      857.0   0.788       0.863          0.725  0.876  0.079               0.084     0.251                  0.267
               39          4  temporada_2010      234.0   0.476       0.359          0.714  0.600  0.042               0.045     0.147                  0.153
               44          4  temporada_2015      557.0   0.480       0.327          0.901  0.811  0.075               0.092     0.244                  0.285
               45          4  temporada_2016      395.0   0.590       0.572          0.608  0.660  0.070               0.070     0.231                  0.228
               46          4  temporada_2017      539.0   0.632       0.590          0.679  0.706  0.081               0.089     0.267                  0.281
               51          4  temporada_2022      431.0   0.652       0.599          0.718  0.743  0.060               0.066     0.200                  0.213
               52          4  temporada_2023     1039.0   0.811       0.736          0.901  0.922  0.081               0.111     0.267                  0.338
               53          4  temporada_2024      857.0   0.761       0.824          0.706  0.840  0.089               0.098     0.279                  0.302
platt_agrupado 3           2  temporada_2010      234.0   0.720       0.697          0.744  0.780  0.030               0.033     0.110                  0.114
               8           2  temporada_2015      557.0   0.771       0.693          0.867  0.885  0.051               0.065     0.174                  0.203
               9           2  temporada_2016      395.0   0.656      

In [10]:
pd.concat({k: v["semanas_brote_040_050"] for k, v in rd.items()}, names=["variante"]).round(1)


horizonte          bloque  brote_en_040_050  semanas_brote  brote_en_040_050_rango  semanas_brote_rango
variante                                                                                                                  
referencia     0           2  temporada_2010              15.0          234.0                      10                    0
               1           2  temporada_2015              54.0          557.0                       7                    0
               2           2  temporada_2016              35.0          395.0                       6                    0
               3           2  temporada_2017              47.0          539.0                       4                    0
               4           2  temporada_2022              34.0          431.0                       4                    0
               5           2  temporada_2023              54.0         1039.0                       5                    0
               6           2  temporada_2024              32.0          857.0                      11                    0
               7           3  temporada_2010              33.0          234.0                       5                    0
               8           3  temporada_2015              93.0          557.0                       7                    0
               9           3  temporada_2016              36.0          395.0                       5                    0
               10          3  temporada_2017              67.0          539.0                       4                    0
               11          3  temporada_2022              36.0          431.0                       3                    0
               12          3  temporada_2023              57.0         1039.0                       8                    0
               13          3  temporada_2024              21.0          857.0                       5                    0
               14          4  temporada_2010              45.0          234.0                       9                    0
               15          4  temporada_2015             129.0          557.0                       7                    0
               16          4  temporada_2016              49.0          395.0                       6                    0
               17          4  temporada_2017              71.0          539.0                       3                    0
               18          4  temporada_2022              45.0          431.0                       5                    0
               19          4  temporada_2023              80.0         1039.0                       2                    0
               20          4  temporada_2024              36.0          857.0                       5                    0
platt_agrupado 0           2  temporada_2010              16.0          234.0                       9                    0
               1           2  temporada_2015              52.0          557.0                       5                    0
               2           2  temporada_2016              37.0          395.0                       3                    0
               3           2  temporada_2017              35.0          539.0                       4                    0
               4           2  temporada_2022              20.0          431.0                       2                    0
               5           2  temporada_2023              33.0         1039.0                       5                    0
               6           2  temporada_2024              39.0          857.0                       8                    0
               7           3  temporada_2010              30.0          234.0                       5                    0
               8           3  temporada_2015              92.0          557.0                       4                    0
               9           3  temporada_2016              36.0          395.0      

In [11]:
pd.concat({k: v["recall_por_tipo"] for k, v in rd.items()}, names=["variante"]).round(3)


horizonte          bloque         tipo  semanas  recall  semanas_rango  recall_rango
variante                                                                                               
referencia     4           2  temporada_2010  brote_nuevo     51.0   0.118              0         0.059
               5           2  temporada_2010     en_curso    183.0   0.863              0         0.033
               14          2  temporada_2015  brote_nuevo     60.0   0.067              0         0.000
               15          2  temporada_2015     en_curso    497.0   0.771              0         0.014
               16          2  temporada_2016  brote_nuevo     53.0   0.132              0         0.000
               17          2  temporada_2016     en_curso    342.0   0.804              0         0.003
               18          2  temporada_2017  brote_nuevo     79.0   0.329              0         0.000
               19          2  temporada_2017     en_curso    460.0   0.815              0         0.004
               26          2  temporada_2022  brote_nuevo     49.0   0.122              0         0.041
               27          2  temporada_2022     en_curso    382.0   0.770              0         0.005
               28          2  temporada_2023  brote_nuevo     86.0   0.174              0         0.000
               29          2  temporada_2023     en_curso    953.0   0.887              0         0.001
               30          2  temporada_2024  brote_nuevo     34.0   0.353              0         0.000
               31          2  temporada_2024     en_curso    823.0   0.874              0         0.004
               36          3  temporada_2010  brote_nuevo     51.0   0.059              0         0.000
               37          3  temporada_2010     en_curso    183.0   0.689              0         0.022
               46          3  temporada_2015  brote_nuevo     60.0   0.033              0         0.000
               47          3  temporada_2015     en_curso    497.0   0.573              0         0.010
               48          3  temporada_2016  brote_nuevo     53.0   0.113              0         0.038
               49          3  temporada_2016     en_curso    342.0   0.743              0         0.003
               50          3  temporada_2017  brote_nuevo     79.0   0.228              0         0.000
               51          3  temporada_2017     en_curso    460.0   0.739              0         0.009
               58          3  temporada_2022  brote_nuevo     49.0   0.163              0         0.000
               59          3  temporada_2022     en_curso    382.0   0.749              0         0.010
               60          3  temporada_2023  brote_nuevo     86.0   0.186              0         0.000
               61          3  temporada_2023     en_curso    953.0   0.849              0         0.006
               62          3  temporada_2024  brote_nuevo     34.0   0.265              0         0.000
               63          3  temporada_2024     en_curso    823.0   0.888              0         0.005
               68          4  temporada_2010  brote_nuevo     51.0   0.020              0         0.000
               69          4  temporada_2010     en_curso    183.0   0.454              0         0.049
               78          4  temporada_2015  brote_nuevo     60.0   0.017              0         0.000
               79          4  temporada_2015     en_curso    497.0   0.364              0         0.012
               80          4  temporada_2016  brote_nuevo     53.0   0.057              0         0.000
               81          4  temporada_2016     en_curso    342.0   0.652              0         0.006
               82          4  temporada_2017  brote_nuevo     79.0   0.203              0         0.013
               83          4  temporada_2017     en_curso    460.0   0.654              0         0.007
               90          4  temporada_2022  brote_nuevo     49.0 

In [12]:
pd.concat({k: v["hu0007_4"] for k, v in rd.items()}, names=["variante"])


horizonte          bloque  precision_050  recall_050    f1_050  razon_mae  razon_rmse  cumple_precision_050  cumple_recall_050  cumple_f1_050  cumple_razon_mae  cumple_razon_rmse  \
variante                                                                                                                                                                                                
referencia     0           2  temporada_2010       0.745455    0.700855  0.722467   0.938085    0.919739                  True              False           True             False              False   
               1           2  temporada_2015       0.865169    0.694794  0.772455   0.945156    1.053429                  True              False           True             False              False   
               2           2  temporada_2016       0.609071    0.713924  0.657343   1.100044    1.071417                  True              False          False             False              False   
               3           2  temporada_2017       0.663921    0.743970  0.703316   0.800489    0.875387                  True              False           True              True              False   
               4           2  temporada_2022       0.728155    0.696056  0.711905   0.961816    1.010115                  True              False           True             False              False   
               5           2  temporada_2023       0.888545    0.827719  0.857428   0.846863    0.940025                  True               True           True              True              False   
               6           2  temporada_2024       0.752058    0.852975  0.799344   1.259388    1.565843                  True               True           True             False              False   
               7           3  temporada_2010       0.731844    0.551282  0.630807   0.896763    0.890730                  True              False          False             False              False   
               8           3  temporada_2015       0.875000    0.515260  0.648588   0.894666    1.044787                  True              False          False             False              False   
               9           3  temporada_2016       0.599078    0.658228  0.627262   1.130717    1.142470                 False              False          False             False              False   
               10          3  temporada_2017       0.652095    0.664193  0.658065   0.754253    0.840480                  True              False          False              True               True   
               11          3  temporada_2022       0.727723    0.682135  0.704327   0.959157    1.063237                  True              False           True             False              False   
               12          3  temporada_2023       0.880342    0.794033  0.835020   0.790613    0.887695                  True              False           True              True              False   
               13          3  temporada_2024       0.725221    0.863477  0.787814   1.382340    1.777912                  True               True           True             False              False   
               14          4  temporada_2010       0.714286    0.358974  0.475921   0.875766    0.858832                  True              False          False             False              False   
               15          4  temporada_2015       0.900990    0.326750  0.479578   0.846978    0.993067                  True              False          False              True              False   
               16          4  temporada_2016       0.608108    0.572152  0.590377   1.150254    1.175559                  True              False          False             False              False   
               17          4  temporada_2017       0.679487    0.589981  0.631579   0.729969    0.812555                  True              False          False              True               True   

In [13]:
pd.concat({k: v["antelacion_semilla_17"] for k, v in rd.items()}, names=["variante"]).round(3)


horizonte          bloque               alerta  episodios  avisados_2_o_mas  ic95_inferior  ic95_superior
variante                                                                                                                     
referencia     2            2  temporada_2010           alerta_050         30             0.033          0.000          0.100
               7            2  temporada_2015           alerta_050         34             0.059          0.000          0.147
               8            2  temporada_2016           alerta_050         34             0.206          0.088          0.353
               9            2  temporada_2017           alerta_050         56             0.446          0.321          0.571
               13           2  temporada_2022           alerta_050         38             0.132          0.026          0.237
               14           2  temporada_2023           alerta_050         57             0.193          0.105          0.298
               15           2  temporada_2024           alerta_050         23             0.435          0.217          0.652
               19           3  temporada_2010           alerta_050         30             0.067          0.000          0.167
               24           3  temporada_2015           alerta_050         34             0.029          0.000          0.088
               25           3  temporada_2016           alerta_050         34             0.265          0.118          0.412
               26           3  temporada_2017           alerta_050         56             0.446          0.321          0.571
               30           3  temporada_2022           alerta_050         38             0.184          0.079          0.316
               31           3  temporada_2023           alerta_050         57             0.316          0.211          0.439
               32           3  temporada_2024           alerta_050         23             0.565          0.348          0.783
               36           4  temporada_2010           alerta_050         30             0.033          0.000          0.100
               41           4  temporada_2015           alerta_050         34             0.029          0.000          0.088
               42           4  temporada_2016           alerta_050         34             0.176          0.059          0.324
               43           4  temporada_2017           alerta_050         56             0.411          0.286          0.536
               47           4  temporada_2022           alerta_050         38             0.158          0.053          0.289
               48           4  temporada_2023           alerta_050         57             0.368          0.246          0.491
               49           4  temporada_2024           alerta_050         23             0.522          0.304          0.739
               53           2  temporada_2010  alerta_persistencia         30             0.033          0.000          0.100
               58           2  temporada_2015  alerta_persistencia         34             0.059          0.000          0.147
               59           2  temporada_2016  alerta_persistencia         34             0.118          0.029          0.235
               60           2  temporada_2017  alerta_persistencia         56             0.161          0.071          0.250
               64           2  temporada_2022  alerta_persistencia         38             0.105          0.026          0.211
               65           2  temporada_2023  alerta_persistencia         57             0.123          0.053          0.211
               66           2  temporada_2024  alerta_persistencia         23             0.130          0.000          0.261
               70           3  temporada_2010  alerta_persistencia         30             0.067          0.000          0.167
               75           3  temporada_2015  alerta_persistencia         34             0.059          0.000    

## 5 · Salidas: métricas y MLflow


In [14]:
to_json = lambda d: {k: diag.to_json(v) for k, v in d.items() if isinstance(v, pd.DataFrame)}
content = {
    "guardas": {"G5": g5, "G6": g6},
    "paso_A": {**to_json(step_a), "guarda_calibracion": {k: v.to_dict() for k, v in guard_a.items()},
               "adopta_1e-15": adopt_a},
    "paso_B": {k: to_json(v) for k, v in step_b.items()},
    "guarda_calibracion_B": {k: v.to_dict() for k, v in guard_b.items()},
    "adoptadas_B": list(adopted), "cruces": crosses, "final": {"calibrador": final, "recorte": eps},
    "lectura_sin_2010": {k: diag.to_json(v) for k, v in reading_no_2010.items()},
    "calibradores_semilla_17": diag.to_json(cal),
    "lecturas": {k: to_json(v) for k, v in rd.items()},
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
verdicts = {"A": step_a["veredicto"], **{f"B.{k}": v["veredicto"] for k, v in step_b.items()}}
tracking.log_study(
    "mejora_modelo_fase2_calibracion", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "2", "preregistro": "3bbfa10", "final": final, "recorte": str(eps),
                       **{f"veredicto.{k}": "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in v.itertuples())
                          for k, v in verdicts.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={f"veredicto_{k}": v for k, v in verdicts.items()},
    description="Fase 2 del estudio 06: variantes de calibración sobre L.")


C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio d320f5c3d1e54e529b36901f3e3a0541 en 'denguekay/mejora_modelo_fase2_calibracion' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'd320f5c3d1e54e529b36901f3e3a0541'